# Evaluation and Ablation Studies

Evaluate classifier performance and run ablation studies.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent))

import json
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from src.models.ml_step_classifier import MLStepClassifierWrapper
from src.data.loaders import LABEL_TO_ID, ID_TO_LABEL
import torch


In [ ]:
# Load test data
with open("data/processed/test.json", 'r') as f:
    test_data = json.load(f)

print(f"Test examples: {len(test_data)}")


In [ ]:
# Load model
classifier = MLStepClassifierWrapper(
    model_path="models/checkpoints/",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Evaluate
predictions = []
labels = []

for item in test_data[:100]:  # Sample for speed
    result = classifier.infer(
        problem=item['problem'],
        prev_steps=item.get('prev_steps_context', ''),
        current_step=item['step_text']
    )
    predictions.append(LABEL_TO_ID[result['label']])
    labels.append(LABEL_TO_ID[item['label']])


In [ ]:
# Compute metrics
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    labels, predictions, average='macro', zero_division=0
)

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")


In [ ]:
# Confusion matrix
cm = confusion_matrix(labels, predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[ID_TO_LABEL[i] for i in range(len(ID_TO_LABEL))],
            yticklabels=[ID_TO_LABEL[i] for i in range(len(ID_TO_LABEL))])
plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()
